<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_11_Model_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers

In [2]:
import torch
import time
import os

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("Original model loaded successfully!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Original model loaded successfully!


In [4]:
def get_model_size(model):
    total_size = 0

    for param in model.parameters():
        total_size += param.numel() * param.element_size()

    return total_size / (1024 ** 2)


original_size = get_model_size(model)

print("Original Model Size: {:.2f} MB".format(original_size))

Original Model Size: 255.41 MB


In [5]:
test_text = "Artificial Intelligence can help improve healthcare systems."

inputs = tokenizer(
    test_text,
    return_tensors="pt"
)

print("Input prepared successfully!")

Input prepared successfully!


In [6]:
model.eval()

start_time = time.time()

with torch.no_grad():
    original_output = model(**inputs)

end_time = time.time()

original_time = end_time - start_time

print("Original Model Inference Time: {:.6f} seconds".format(original_time))

Original Model Inference Time: 0.415858 seconds


In [7]:
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Dynamic quantization applied successfully!")

/tmp/ipykernel_7811/771520878.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Dynamic quantization applied successfully!


In [8]:
quantized_size = get_model_size(quantized_model)

print("Original Model Size: {:.2f} MB".format(original_size))
print("Quantized Model Size: {:.2f} MB".format(quantized_size))

Original Model Size: 255.41 MB
Quantized Model Size: 91.00 MB


In [9]:
quantized_model.eval()

start_time = time.time()

with torch.no_grad():
    quantized_output = quantized_model(**inputs)

end_time = time.time()

quantized_time = end_time - start_time

print("Quantized Model Inference Time: {:.6f} seconds".format(quantized_time))

Quantized Model Inference Time: 0.184473 seconds


In [10]:
print("========== MODEL OPTIMIZATION COMPARISON ==========")

print("\nOriginal Model")
print("Size: {:.2f} MB".format(original_size))
print("Inference Time: {:.6f} seconds".format(original_time))

print("\nQuantized Model")
print("Size: {:.2f} MB".format(quantized_size))
print("Inference Time: {:.6f} seconds".format(quantized_time))

size_reduction = ((original_size - quantized_size) / original_size) * 100

print("\nSize Reduction: {:.2f}%".format(size_reduction))

========== MODEL OPTIMIZATION COMPARISON ==========

Original Model
Size: 255.41 MB
Inference Time: 0.415858 seconds

Quantized Model
Size: 91.00 MB
Inference Time: 0.184473 seconds

Size Reduction: 64.37%


In [11]:
print("========== OUTPUT VERIFICATION ==========")

print("\nOriginal Model Output:")
print(original_output.logits)

print("\nQuantized Model Output:")
print(quantized_output.logits)

========== OUTPUT VERIFICATION ==========

Original Model Output:
tensor([[-0.1492,  0.0160]])

Quantized Model Output:
tensor([[-0.0698,  0.0240]])
